# Project 20: Data Quality, Drift & Pipeline Observability

**Simple college project:** A program that checks monthly shop sales data for missing values, duplicates, invalid values, changes, and late data.

## Cell 1 — Import libraries
Libraries are tools that help Python work with tables and numbers.

In [ ]:
import pandas as pd
import numpy as np


## Cell 2 — Create sample sales data
We make imaginary sales data for four months. Each month has 100 rows.

In [ ]:
np.random.seed(10)

def make_data(month, rows=100):
    dates = pd.date_range(f"2026-{month}-01", periods=rows, freq="6h")
    data = {
        "transaction_id": [f"{month}-{i}" for i in range(rows)],
        "date": dates,
        "product": np.random.choice(["Laptop", "Shirt", "Rice", "Chair"], rows),
        "category": np.random.choice(["Electronics", "Clothing", "Food", "Furniture"], rows),
        "quantity": np.random.randint(1, 10, rows),
        "price": np.random.randint(100, 5000, rows),
        "city": np.random.choice(["Delhi", "Mumbai", "Pune", "Jaipur"], rows)
    }
    return pd.DataFrame(data)

jan = make_data("01")
feb = make_data("02")
mar = make_data("03")
apr = make_data("04")

print("January rows:", len(jan))
print(jan.head())

## Cell 3 — Add some problems on purpose
We change March prices, and add mistakes to April so our checker can find them.

In [ ]:
# March: prices changed (drift)
mar["price"] = mar["price"] * 1.5

# April: missing city values
apr.loc[0:4, "city"] = np.nan

# April: duplicate rows
apr = pd.concat([apr, apr.iloc[0:5]], ignore_index=True)

# April: invalid quantities (zero is not allowed)
apr.loc[5:7, "quantity"] = 0

# April: new column added
apr["discount"] = 10

# April: category name changed
apr["category"] = apr["category"].replace("Food", "Grocery")

print("April data now has", len(apr), "rows")

## Cell 4 — Check missing values and duplicates

In [ ]:
def basic_check(data, month):
    print("\n---", month, "---")
    print("Rows:", len(data))
    print("Missing values per column:")
    print(data.isnull().sum())
    print("Duplicate transaction IDs:", data["transaction_id"].duplicated().sum())

basic_check(jan, "January")
basic_check(feb, "February")
basic_check(mar, "March")
basic_check(apr, "April")

## Cell 5 — Check invalid values

In [ ]:
def check_invalid(data):
    print("Quantity zero or below:", (data["quantity"] <= 0).sum())
    print("Price zero or below:", (data["price"] <= 0).sum())

check_invalid(apr)

## Cell 6 — Check schema (columns) changes

In [ ]:
expected_columns = [
    "transaction_id", "date", "product", "category",
    "quantity", "price", "city"
]

def check_columns(data):
    missing = [c for c in expected_columns if c not in data.columns]
    extra = [c for c in data.columns if c not in expected_columns]
    print("Missing columns:", missing)
    print("New / extra columns:", extra)

check_columns(apr)

## Cell 7 — Compare price average and detect drift
Drift means the data pattern has changed. Here we compare average prices in January and March.

In [ ]:
jan_average = np.mean(jan["price"])
mar_average = np.mean(mar["price"])
change_percent = ((mar_average - jan_average) / jan_average) * 100

print("January average price:", round(jan_average, 2))
print("March average price:", round(mar_average, 2))
print("Change percent:", round(change_percent, 2), "%")

if abs(change_percent) > 20:
    print("ALERT: Price drift detected!")
else:
    print("No major price drift.")

## Cell 8 — Compare monthly data volume

In [ ]:
months = ["January", "February", "March", "April"]
counts = [len(jan), len(feb), len(mar), len(apr)]

for i in range(1, len(months)):
    change = ((counts[i] - counts[i-1]) / counts[i-1]) * 100
    print(months[i-1], "to", months[i], "change:", round(change, 2), "%")
    if abs(change) > 30:
        print("ALERT: Unusual data volume change!")

## Cell 9 — Simple number summary using NumPy
We calculate average, middle value (median), and how spread out prices are (standard deviation).

In [ ]:
print("January price summary")
print("Average:", round(np.mean(jan["price"]), 2))
print("Median:", round(np.median(jan["price"]), 2))
print("Standard deviation:", round(np.std(jan["price"]), 2))

## Cell 10 — Compare two monthly snapshots
We compare January and April to see what columns were added or missing.

In [ ]:
jan_columns = set(jan.columns)
apr_columns = set(apr.columns)
print("Columns added:", list(apr_columns - jan_columns))
print("Columns missing:", list(jan_columns - apr_columns))
print("January rows:", len(jan))
print("April rows:", len(apr))

## Cell 11 — Check data freshness (late data)
We expect April data up to April 30. If the latest date is more than 3 days earlier, show an alert.

In [ ]:
expected_date = pd.Timestamp("2026-04-30")
latest_date = apr["date"].max()
delay_days = (expected_date - latest_date).days
print("Latest date in data:", latest_date)
print("Days behind:", delay_days)
if delay_days > 3:
    print("ALERT: Data may be late!")
else:
    print("Data freshness looks okay.")

## Cell 12 — Check product reference list
A reference list contains the product names we expect to see.

In [ ]:
allowed_products = ["Laptop", "Shirt", "Rice", "Chair"]
wrong_products = apr[~apr["product"].isin(allowed_products)]
print("Products not in reference list:", wrong_products["product"].unique())
print("Number of unknown products:", len(wrong_products))

## Cell 9 — Make a simple alert report

In [ ]:
alerts = []

if apr.isnull().sum().sum() > 0:
    alerts.append("WARNING: Missing values found in April")
if apr["transaction_id"].duplicated().sum() > 0:
    alerts.append("WARNING: Duplicate transaction IDs found in April")
if (apr["quantity"] <= 0).sum() > 0:
    alerts.append("WARNING: Invalid quantity found in April")
if "discount" in apr.columns:
    alerts.append("INFO: New column discount found in April")
if "Grocery" in apr["category"].values:
    alerts.append("WARNING: New category Grocery found")
if delay_days > 3:
    alerts.append("WARNING: April data appears late")
if abs(change_percent) > 20:
    alerts.append("WARNING: Price drift found between January and March")

print("MONITORING ALERT REPORT")
for alert in alerts:
    print("-", alert)

## Cell 10 — Save report files
This creates CSV files that can be downloaded from Colab’s Files panel.

In [ ]:
pd.DataFrame({"Alert": alerts}).to_csv("alert_report.csv", index=False)

monthly_summary = pd.DataFrame({"Month": months, "Rows": counts})
monthly_summary.to_csv("monthly_summary.csv", index=False)

print("Saved alert_report.csv and monthly_summary.csv")

## Cell 11 — Conclusion
This project checks sales data for missing values, duplicates, invalid values, schema changes, price drift, and unusual monthly record counts. It prints alerts to help a user identify data problems early.